# Exploratory Data Analysis (EDA) - ADN Pipeline
Notebook này sử dụng kiến trúc OOP (`CocoEDAAnalyzer`) để trích xuất 7 biểu đồ phân tích chuyên sâu cho các tập dữ liệu rác thải (TACO và Roboflow).

In [ ]:
import os
import sys
import json
from pathlib import Path

# Đảm bảo import được module từ src/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.waste_detection.data.eda_analyzer import CocoEDAAnalyzer

## 1. Phân tích tập dữ liệu TACO (Đã ánh xạ 7 nhãn)
Khởi tạo Analyzer và cấu hình nhãn.

In [ ]:
# Đường dẫn tới dữ liệu TACO
taco_ann_path = os.path.join(project_root, 'data', 'raw', 'TACO', 'annotations.json')
taco_img_dir = os.path.join(project_root, 'data', 'raw', 'TACO')

# Đọc cấu hình ánh xạ từ 60 nhãn về 7 nhãn
mapping_file = os.path.join(project_root, 'configs', 'data', 'mapping_label.json')
with open(mapping_file, 'r', encoding='utf-8') as f:
    mapping_dict = json.load(f)

original_to_target = {}
for target_name, original_names in mapping_dict.items():
    for orig in original_names:
        original_to_target[orig] = target_name

def taco_mapper(cat_id, cat_name):
    mapped = original_to_target.get(cat_name)
    if mapped == "ignore" or mapped is None:
        return None
    return mapped

try:
    # Khởi tạo Analyzer
    taco_analyzer = CocoEDAAnalyzer(
        annotation_file=taco_ann_path,
        image_dir=taco_img_dir,
        dataset_name="TACO_7Class",
        label_mapper=taco_mapper
    )
    print("✅ Khởi tạo Analyzer thành công!")
except FileNotFoundError:
    print(f"❌ Không tìm thấy dữ liệu TACO tại {taco_ann_path}. Vui lòng tải dữ liệu về máy trước.")
    taco_analyzer = None

### 1.1 Phân bố số lượng rác theo từng lớp

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_class_distribution()

### 1.2 Phân bố kích thước Bounding Box (Width x Height)

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_bbox_size_distribution()

### 1.3 Histogram: Số lượng vật thể rác trên mỗi ảnh

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_objects_per_image()

### 1.4 Heatmap Không gian: Rác thường nằm ở đâu trong khung hình?

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_bbox_heatmap()

### 1.5 Tỉ lệ khung hình (Aspect Ratio W/H) - Quan trọng cho Anchor Box YOLO

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_aspect_ratio_distribution()

### 1.6 Ma trận đồng xuất hiện (Co-occurrence Matrix)

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_class_cooccurrence()

### 1.7 Vẽ thử Bounding Box lên một vài ảnh mẫu

In [ ]:
if taco_analyzer:
    taco_analyzer.visualize_sample_images(num_samples=3)

---
## 2. Phân tích tập dữ liệu Roboflow (Nếu có)
Chỉ cần truyền đường dẫn của tập Roboflow vào bộ khởi tạo `CocoEDAAnalyzer` tương tự như trên.

In [ ]:
robo_ann_path = os.path.join(project_root, 'data', 'raw', 'Roboflow', '_annotations.coco.json')
robo_img_dir = os.path.join(project_root, 'data', 'raw', 'Roboflow')

try:
    robo_analyzer = CocoEDAAnalyzer(
        annotation_file=robo_ann_path,
        image_dir=robo_img_dir,
        dataset_name="Roboflow",
        label_mapper=None  # Dữ liệu Roboflow thường đã có 7 nhãn sẵn
    )
    print("✅ Khởi tạo Analyzer thành công!")
    # Bạn có thể gọi các hàm vẽ biểu đồ tương tự như với taco_analyzer ở trên
    robo_analyzer.plot_class_distribution()
except FileNotFoundError:
    print(f"Thông báo: Chưa có dữ liệu Roboflow tại {robo_ann_path}. Bỏ qua bước này.")
    robo_analyzer = None